# 3 · Next-day price prediction & bull/bear classification
Models are trained on the first 80% of history and tested on the last 20% (no shuffling → no look-ahead).
Every model is compared with a **naive baseline** (tomorrow's close = today's close). Beating that baseline on daily data is hard; if a model doesn't, its forecast shouldn't be trusted.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import config
from src import data_loader as dl
OFFLINE = False   # True = synthetic sample data (no internet needed)

In [ ]:
if OFFLINE:
    data = dl.generate_sample_data(config.TICKERS, config.BENCHMARK)
    rf = config.DEFAULT_RISK_FREE
else:
    data = dl.fetch_prices(config.TICKERS + [config.BENCHMARK], config.START_DATE, config.END_DATE)
    rf = dl.fetch_risk_free(config.START_DATE, config.END_DATE)
prices = dl.close_matrix(data, config.TICKERS)
print(f'risk-free rate: {rf:.2%}'); prices.tail()

In [ ]:
from src.forecast import run_forecast
results = {t: run_forecast(data[t], t, data[config.BENCHMARK]) for t in config.TICKERS}
results['AAPL'].regression_metrics

In [ ]:
results['AAPL'].classification_metrics

In [ ]:
pd.DataFrame({t: {**r.signal, **{k: r.next_day[k] for k in ('pred_open', 'pred_close', 'pred_return')}}
              for t, r in results.items()}).T

In [ ]:
tp = results['AAPL'].test_predictions.iloc[-120:]
tp.drop(columns='Naive').plot(figsize=(12, 5), title='AAPL next-day close: predicted vs actual')

In [ ]:
imp = results['AAPL'].feature_importance.head(10)[::-1]
plt.figure(figsize=(8, 4)); plt.barh(imp.index, imp.values); plt.title('Feature importance (RF)')

### Optional: LSTM (needs `pip install tensorflow`)

In [ ]:
# from src.lstm_model import train_lstm
# model, metrics = train_lstm(data['AAPL'], data[config.BENCHMARK], save_path='../models/lstm_aapl.keras')
# metrics